In [ ]:
text_llm = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-7b-chat-hf", ...); tokenizer = AutoTokenizer...
multimodal_llm = load_llava_7b_model(...)
...
output_ids = text_llm.generate(**tokenizer(prompt, return_tensors='pt').to(device))
answer = tokenizer.decode(output_ids[0], skip_special_tokens=True)


In [ ]:
evaluation_queries = [
    {"name": "LLM in a Flash (ACL'24)", "query": "How does the system enable long-context inference on memory-limited devices using NVMe, and what are the key scheduling/prefetching ideas?"},
    {"name": "FlashAttention-2 (2023)", "query": "What kernel-level changes make FlashAttention-2 faster than FlashAttention-1, and how do tiling/parallelization reduce memory traffic?"},
    {"name": "vLLM: PagedAttention (2023)", "query": "Explain how PagedAttention reduces memory fragmentation and improves throughput during batched decoding."},
    {"name": "Speculative Decoding (2023)", "query": "Summarize the draft-verify process and the acceptance criterion in speculative decoding; what governs the trade-off between speedup and quality?"},
    {"name": "H2O: LLM Inference (2023)", "query": "Why is KV-cache eviction in H2O not equivalent to Belady's cache policy, and what heuristics does H2O use?"},
    {"name": "Scaling Laws (Kaplan et al. 2020)", "query": "According to the 2020 paper on scaling laws for neural language models, how do model performance, model size, and dataset size relate?"},
    {"name": "Chinchilla (2022)", "query": "What is the main finding of the Chinchilla paper regarding the allocation of compute between model size and training tokens?"},
    {"name": "RAPTOR (2024)", "query": "How does RAPTOR's hierarchical summarization and retrieval approach improve efficiency compared to standard flat retrieval?"},
    {"name": "GraphRAG (2024)", "query": "What is GraphRAG, and how does it incorporate knowledge graphs into the retrieval-augmented generation pipeline?"},
    {"name": "CRAG (2024)", "query": "What mechanism does CRAG introduce to evaluate retrieval quality and ensure the reliability of generated answers?"}
]


In [ ]:
import time
import pandas as pd

results = []
for mode in ["flat_text", "flat_multi"]:
    for q in evaluation_queries:
        query = q["query"]
        start_time = time.time()
        if mode == "flat_text":
            passages = retrieval.query_text(query, k=3)
            context = "\n".join([p["content"] for p in passages])
            answer = text_llm.generate(f"{context}\nQ: {query}\nA:")
        else:  # flat_multi
            passages = retrieval.query_text(query, k=3)
            images = retrieval.query_images(query, k=2)
            # append captions for simulation.
            context = "\n".join([p["content"] for p in passages])
            for img in images:
                context += "\n[Image: " + img["caption"] + "]"
            answer = multimodal_llm.generate(f"{context}\nQ: {query}\nA:")
        elapsed = time.time() - start_time
        # Dummy quality evaluation
        quality_score = evaluate_answer_quality(answer, q["name"])
        results.append({
            "mode": mode,
            "query_name": q["name"],
            "latency": elapsed,
            "quality": quality_score,
            "answer": answer
        })
        print(f"{mode} | {q['name']} | latency: {elapsed:.2f}s | quality: {quality_score:.2f}")
# Save raw results
pd.DataFrame(results).to_csv("rag_v1/results/flat_eval_results.csv", index=False)


In [ ]:
df = pd.DataFrame(results)
summary = df.groupby("mode").agg(avg_quality=('quality','mean'), avg_latency=('latency','mean'))
print(summary)
summary.to_csv("rag_v1/results/flat_eval_summary.csv")
